In [4]:
# --- BIBLIOTECAS ---
import pandas as pd
import zipfile
import io
import os
import datetime
from tqdm import tqdm
import unicodedata

In [5]:
# --- CONFIGURAÇÕES ---
ROOT_ZIP = r"C:\Users\Mateus Joter\Downloads\download.zip"
LOG_FILE = r"C:\Users\Mateus Joter\Downloads\processamento_chunks_log.txt"
OUTPUT_CSV = "cnpj_brasil_consolidado.csv"
CHUNK_SIZE = 300000  # Processa 300 mil linhas por vez

In [ ]:
# --- FUNÇÕES ---

# Escreve um arquivo de acompanhamento dos processos com data, horário, qntd de linhas lidas, gravadas e não gravadas
def write_log(message, orig=0, grav=0):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    nao_grav = orig - grav
    log_msg = f"[{timestamp}] {message} | Lidas: {orig} | Gravadas: {grav} | Diferença: {nao_grav}"
    with open(LOG_FILE, "a") as f:
        f.write(log_msg + "\n")
    print(log_msg)

# Retira acentos e devolve em caixa alta da string "s"
def strip_accents(s):
    # Verifica se o dado é nulo
    if not isinstance(s, str) or s == "nan" or pd.isna(s):
        return ""

    # Separa a string entre letras e acentos e guarda somente as letras em caixa alta
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                  if unicodedata.category(c) != 'Mn').upper()

# Na preparação dos dados, há uma lógica repetida para os dicionários mais simples, esta função auxilia nisto
def prepare_simple_dics(inner_zip, f_name, name):
    # Verifica o arquivo avaliado
    if f_name.endswith(name + "CSV"):

        # Abre o arquivo
        with inner_zip.open(f_name) as f:
            df = pd.read_csv(f, sep=";", header=None, encoding='latin1', dtype=str)

            # Salva o dicionario
            dicts[name].update(dict(zip(df[0], df[1])))

# Função auxiliar
def simplificar_logradouro(texto):
    if not isinstance(texto, str): return ""
    
    # Simula o startswith/in de forma eficiente
    if "RUA" in texto: return "RUA"
    if "AVENIDA" in texto: return "AVENIDA"
    if "TRAVESSA" in texto: return "TRAVESSA"
    
    return texto # Retorna o original (ex: ALAMEDA, PRACA) se não der match

# Prepara o dicionario de estabelecimentos
def prepare_estabele_dics(inner_zip, f_name):
    # Abre o arquivo
    with inner_zip.open(f_name) as f:
        # Carrega o DataFrame
        df = pd.read_csv(f, sep=";", header=None, encoding='latin1', dtype=str)

        # Novo layout das colunas
        layout_estabelecimento = {
        0: "CNPJ Básico",
        1: "CNPJ Ordem",
        2: "CNPJ DV",
        3: "Matriz/Filial",
        4: "Nome Fantasia",
        5: "Situação Cadastral",
        6: "Data Situação Cadastral",
        7: "Motivo Situação Cadastral",
        8: "Nome da Cidade no Exterior",
        9: "País",
        10: "Data de Início da Atividade",
        11: "CNAE Fiscal Principal",
        12: "CNAE Fiscal Secundária",
        13: "Tipo de Logradouro",
        14: "Logradouro (Nome)",
        15: "Número",
        16: "Complemento",
        17: "Bairro",
        18: "CEP",
        19: "UF",
        20: "Município (Código)",
        21: "DDD 1",
        22: "Telefone 1",
        23: "DDD 2",
        24: "Telefone 2",
        25: "DDD Fax",
        26: "Fax",
        27: "Correio Eletrônico (E-mail)",
        28: "Situação Especial",
        29: "Data da Situação Especial"
        }
        df = df.rename(columns = layout_estabelecimento)
        
        # Corrigindo os numeros dos imoveis
        correcoes_numeros_imoveis = {
            "000": "",
            "00": "",
            "0": "",
            "SN": "S/N"
        }
        df["Número"] = df["Número"].replace(correcoes_imoveis)

        # Corrigindo o tipo de logradouro
        df["Tipo de Logradouro"] = df["Tipo de Logradouro"].apply(simplificar_logradouro)

        # Criando coluna com o cnpj completo
        df["CNPJ"] = df["CNPJ Básico"].astype(str) + df["CNPJ Ordem"].astype(str) + df["CNPJ DV"].astype(str)

        # Puxar nome da cidade do dicionário
        df["Município (Nome)"] = df["Município (Código)"].map(dicts['MUNIC']).fillna("")
        
        # Garantia de que algumas colunas têm strings e remoção de nulos
        cols_concatenar = ["Tipo de Logradouro", "Logradouro (Nome)", "Número", "Município (Nome)", "Bairro", "CEP", "UF", "Município (Código)"]
        for col in cols_concatenar:
            data[col] = data[col].fillna("").astype(str)

        # Matriz ou Filial
        df["Matriz/Filial"] = df["Matriz/Filial"].map({'1': 'Matriz', '2': 'Filial', 1: 'Matriz', 2: 'Filial'}).fillna("outro")

        # Data situação cadastral
        df["Data Situação Cadastral"] = df["Data Situação Cadastral"].str[6:8] + "/" + df["Data Situação Cadastral"].str[4:6] + "/" + df["Data Situação Cadastral"].str[0:4]

        # Data situação especial
        df["Data da Situação Especial"] = df["Data da Situação Especial"].str[6:8] + "/" + df["Data da Situação Especial"].str[4:6] + "/" + df["Data da Situação Especial"].str[0:4]

        # Data de atividade
        df["Data de Início da Atividade"] = df["Data de Início da Atividade"].str[6:8] + "/" + df["Data de Início da Atividade"].str[4:6] + "/" + df["Data de Início da Atividade"].str[0:4]

        # Passando o DataFrame para o dicionário
        dicts["EMPRE"] = df[["CNPJ Básico", "CNPJ", "Matriz/Filial", "Nome Fantasia", "Situação Cadastral", "Data Situação Cadastral", "Motivo Situação Cadastral", \
                             "Nome da Cidade no Exterior", "País", "Data de Início da Atividade", "CNAE Fiscal Principal", "CNAE Fiscal Secundária", \
                             "Tipo de Logradouro", "Logradouro (Nome)", "Número", "Complemento", "Bairro", "CEP", "UF", "Município (Nome)", \
                             "Situação Especial", "Data da Situação Especial" ]]

# Prepara o dicionário de empresas
def prepare_empre_dic(inner_zip, f_name):
     # Abre o arquivo
    with inner_zip.open(f_name) as f:
        # Carrega o DataFrame
        df = pd.read_csv(f, sep=";", header=None, encoding='latin1', dtype=str)

        


# Prepara os dicionários para tratamento dos dados
def prepare_dics():
    # 1. Carregar Tabelas de Apoio (Dicionários) - Precisam estar na RAM por serem chaves de consulta
    # Vamos criar dicionários simples para busca rápida
    dicts = {"CNAE": {}, "NATJU": {}, "QUALS": {}, "PAIS": {}, "MUNIC": {}, "ESTABELE": {}, "EMPRE": {}, "SOCIO": {}}

    write_log("Fase 1: Carregando dicionários e base de Empresas para Memória...")

    # Leitura do .zip de entrada
    with zipfile.ZipFile(ROOT_ZIP, 'r') as root:
        # Listagem dos .zips internos
        inner_zips = [f for f in root.namelist() if f.endswith('.zip')]

        # Loop nos zips internos para tratar o que importa
        for iz_name in tqdm(inner_zips, desc="Lendo Dicionários"):

            # Abre o .zip interno e vê o que tem dentro
            with root.open(iz_name) as z2:
                with zipfile.ZipFile(io.BytesIO(z2.read())) as inner_zip:

                    # Avalia o que está dentro do .zip interno
                    for f_name in inner_zip.namelist():

                        # Loop nos dicionários mais simples
                        for name in ["CNAE", "NATJU", "QUALS", "PAIS", "MUNIC"]:
                            prepare_simple_dics(inner_zip, f_name, name)

                        # Carrega dados de estabelecimentos
                        if f_name.endswith("ESTABELECSV"):
                            prepare_estabele_dics(inner_zip, f_name)

                        # Carrega os dados de empresas
                        elif f_name.endswith("EMPRECSV"):
                            with inner_zip.open(f_name) as f:
                                # Aqui carregamos apenas ID, Razão Social e Porte para economizar RAM
                                df = pd.read_csv(f, sep=";", header=None, encoding='latin1', dtype=str, usecols=[0, 1, 5])
                                # Criamos um dataframe de referência rápido
                                dicts["EMPRE"] = df.rename(columns={0:'id_cnpj', 1:'razao', 5:'porte'})

In [12]:
dados = pd.read_csv(r"E:\PROJETO VITAE - GEOPROCESSAMENTO e MODELAGEM\PYTHON\F.K03200$Z.D60214.CNAECSV", sep=";", header=None, encoding='latin1', dtype=str)

teste = {'uau': {}}

teste['uau'].update(dict(zip(dados[0], dados[1])))